**Машинное обучение в экономике**

**Семинар 5. Машинное обучение в эконометрике**

Установка библиотек

In [ ]:
# !python.exe -m pip install --upgrade pip
# !pip install numpy
# !pip install pandas
# !pip install scikit-learn
# !pip install openpyxl
# !pip install doubleml

In [ ]:
# Подключим необходимые библиотеки
import numpy as np                                        # базовые операции с массивами
import pandas as pd                                       # базовые операции с датафреймами
import scipy as scipy
from copy import deepcopy
import math
from scipy.stats import multivariate_normal
import seaborn
import doubleml as dml
from sklearn.ensemble import RandomForestClassifier       # случайный лес (классификация)
from sklearn.ensemble import RandomForestRegressor        # случайный лес (регрессия)
from sklearn.ensemble import GradientBoostingClassifier   # градиентный бустинг (классификация)
from sklearn.ensemble import GradientBoostingRegressor    # градиентный бустинг (регрессия)
from sklearn.linear_model import Lasso                    # Лассо
from sklearn.base import clone
import statsmodels.api as sm                              # линейная регрессия

**Симуляция данных** 🐱

Цели ⭐

*   Симулировать переменные, используя процесс генерации данных, удовлеворяющий предпосылками модели двойного машинного обучения с инструментальными переменными.

При анализе реальных данных, мы можем оценить качество модели, например, благодаря измерению точности прогнозов на тестовой выборке или с помощью кросс-валидации. Кроме того, качество модели в эконометрике часто измеряется при помощи информационных критериев, таких как AIC и BIC.

**Проблема** - эти критерии оценивают качество (обычно прогностическое) модели в целом, однако, зачастую, малоинформативны о том, насколько хорошо эти модели измеряют влияние переменной воздействия $T_{i}$ на целевую переменную $Y_{i}$.

**Решение** - для того, чтобы в учебных целях иметь возможность сравнивать качество не прогнозов, а оценок параметров моделей, воспользуемся не реальными данными, а симулированными. То есть сами создадим данные, что позволит нам заранее знать истинные значения интересующих нас параметров, отражающих связи между переменными. Затем мы сможем сравнить, насколько хорошо различные модели сравнивают эти коэффициенты и как меняется качество этих оценок в зависимости от соблюдения предпосылок методов.

Предположим, что заработная плата $Y_{i}=\text{Wage}_{i}$ зависит от таких признаков $X_{i}$ как:


*   $\text{Educ}_{i}$ - факт наличя высшего образования ($1$ - есть, $0$ - нет).
*   $\text{Experience}_{i}$ - опыт работы.
*   $\text{Health}_{i}$ - здоровье.
*   $\text{Abilities}_{i}$ - способности.

Также, в данных имеется информация об образовании родителей $\text{Parents}_{i}$ ($1$ - высшее, $0$ - нет), которое будет использоваться в качестве инструментальной переменной.

Для наглядной демонстрации принципа работы рассматриваемых методов симулируем, то есть создадим искусственно данные, начав с генерации признаков.

Опишем процесс генерации данных в общем виде. При этом будем рассматривать процесс генерации данных, предполагаемый двойным машинным обучением (ДМО / DML).

Целевая переменная:

$$\text{Wage}_{i} = \alpha \text{Educ}_{i} + g(\text{Experience}_{i},\text{Health}_{i}) + \varepsilon_{i}^{\text{Wage}}$$

Переменная воздействия:

$$\text{Educ}_{i} = g_{\text{Educ}}(\text{Experience}_{i},\text{Health}_{i}) + \varepsilon_{i}^{\text{Educ}}$$

Инструментальная переменная:

$$\text{Parents}_{i} = g_{\text{Parents}}(\text{Experience}_{i},\text{Health}_{i}) + \varepsilon_{i}^{\text{Parents}}$$


В данном случае нас не интересует получение точных прогнозов заработной платы $\text{Wage}_{i}$.

**Цель** - получить как можно более точную оценку параметра $\alpha$, отражающего влияние образования $\text{Educ}_{i}$ на заработную плату $\text{Wage}_{i}$.

**Интерпретация** - получение высшего образования $\text{Educ}_{i} = 1$, при прочем равном стаже и здоровье, повышает заработную плату $\text{Wage}_{i}$ на $\alpha$.

**Допущения о распределении**

Все переменные и случайные ошибки предполагаются независимыми и одинаково распределенными между наблюдениями, то есть по $i\in\{1,...n\}$, где $n$ - число наблюдений.

$$\text{E}(\varepsilon^{\text{Wage}}_{i}|\text{Experience}_{i}, \text{Health}_{i},\text{Parents}_{i})=0$$

$$\text{E}(\varepsilon^{\text{Educ}}_{i}|\text{Experience}_{i}, \text{Health}_{i})=0$$

$$\text{E}(\varepsilon^{\text{Parents}}_{i}|\text{Experience}_{i}, \text{Health}_{i})=0$$

In [ ]:
# Число наблюдений
n = 10000

# Для воспроизводимости
np.random.seed(123)

Симулируем контрольные переменные $\text{Experience}_{i}$, $\text{Health}_{i}$ и $\text{Abilities}_{i}$ из нормального распределения с математическим ожиданием $\mu=25$ и стандартным отклонением $\sigma=10$, то есть $\text{N}\left(25, 10^2\right)$. Для удобства также усечем это распределение сверху и снизу числами $1$ и $50$ соответственно, а после этого округлим полученные значения.

Техническое примечание ⚡

Функция `scipy.stats.norm.rvs` генерирует `size` реализаций случайных величин из нормального распределения с математическим ожиданием `loc` и стандартным отклонением `scale`.

In [ ]:
# Опыт работы
experience = scipy.stats.norm.rvs(size = n,
                                  loc = 25, scale = 10)   # генерация
experience[experience >= 50] = 50                         # усечение
experience[experience <= 1] = 1
experience = np.round(experience)                         # округление
print({'experience': experience[0:10]})                   # первые 10 наблюдений

# Здоровье
health = scipy.stats.norm.rvs(size = n,                   # генерация
                              loc = 25, scale = 10)
health[health >= 50] = 50                                 # усечение
health[health <= 1] = 1
health = np.round(health)                                 # округление
print({'health': health[0:10]})                           # первые 10 наблюдений

# Способности
abilities = scipy.stats.norm.rvs(size = n,
                                 loc = 25, scale = 10)    # генерация
abilities[abilities >= 50] = 50                           # усечение
abilities[abilities <= 1] = 1
abilities = np.round(abilities)                           # округление
print({'abilities': abilities[0:10]})                     # первые 10 наблюдений

Применим фантазию 🦄 и симулируем инструментальную переменную: уровень образования родителей $\text{Parents}_{i}$.

Для краткости обозначим $X_{i}=(\text{Experience}_{i}, \text{Health}_{i})$.

Необходимо симулировать переменную $\text{Parents}_{i}$ так, чтобы она была представима в виде:

$$\text{Parents}_{i} = g_{\text{Parents}}(X_{i}) + \varepsilon^{\text{Parents}}_{i} $$

При этом необходимо соблюсти:

$$\text{E}(\varepsilon^{\text{Parents}}_{i}|X_{i})=0$$

Для этого рассмотрим:

$$\varepsilon^{\text{Parents}}_{i} = \text{Parents}_{i} - \text{E}(\text{Parents}_{i}|X_{i})$$

Действительно, в таком случае $\text{E}(\varepsilon^{\text{Parents}}_{i}|X_{i})=0$, поскольку:

$$\text{E}(\varepsilon^{\text{Parents}}_{i}|X_{i}) = \text{E}(\text{Parents}_{i}|X_{i}) - \text{E}(\text{E}(\text{Parents}_{i}|X_{i})|X_{i})=\text{E}(\text{Parents}_{i}|X_{i})-\text{E}(\text{Parents}_{i}|X_{i})=0$$

Из полученного результата следует, что $g_{\text{Parents}}(X_{i}) = \text{E}(\text{Parents}_{i}|X_{i})$ является достаточным условием для $E(\varepsilon^{\text{Parents}}_{i}|X_{i})=0$, что мотивирует запись:

$$\text{Parents}_{i} = \underbrace{\text{E}(\text{Parents}_{i}|X_{i})}_{g_{\text{Parents}}(X_{i})} + \varepsilon^{\text{Parents}}_{i} $$

**Проблема** - из полученного результата все еще не ясно, как генерировать $\text{Parents}_{i}$ таким образом, чтобы получилась бинарная переменная.

**Решение** - понять, как в данном случае будут выглядеть условные вероятности, зная которые мы можем генерировать значения $\text{Parents}_{i}$ с соответствующими вероятностями.

Обратим внимание, что поскольку $\text{Parents}_{i}$ является бинарной переменной, то имеет распределение Бернулли, а занчит условное математическое ожидание и условная вероятность совпадают:

$$\text{E}(\text{Parents}_{i}|X_{i}) = 1\times P(\text{Parents}_{i} = 1|X_{i}) + 0\times P(\text{Parents}_{i} = 0|X_{i}) = P(\text{Parents}_{i} = 1|X_{i})$$

Следовательно, функция $g_{\text{Parents}}(X_{i})$ будет совпадать с условными вероятностями и можно рассмотреть, например, следующую спецификацию:

$$g_{\text{Parents}}(X_{i}) = P(\text{Parents}_{i} = 1|X_{i}) = \underbrace{F_{\text{Student}}\left(\ln\left(\text{Health}_{i}+\text{Experience}_{i}\right) - 4.3\right)}_{\text{из воображения}}$$

Где $F_{\text{Student}}$ - функция распределения, относящаяяся к распределению Стьюдента с $5$-ю степенями свободы. Она гарантирует, что все вероятности будут в диапазоне от $0$ до $1$.

**Общий алгоритм геренации** $\text{Parents}_{i}$

*   Берем любую функцию, принимающую значения от $0$ до $1$, например, функцию распределения любого распределения $F$. Для удобства лучше использовать непрерывные распределения с носителем на $R$.
*   В качестве аргумента этой функции рассматривается любая (желательно дифференцируемая) функция от признаков $h(X_{i})$.
*   В результате получаем условные вероятности $P\left(\text{Parents}_{i}=1|X_{i}\right)=F(h(X_{i}))$, с которыми затем генерируются значения переменной $\text{Parents}_{i}$.
*  Смотрим на долю $1$ переменной $\text{Parents}_{i}$, чтобы она равнялась адекватному (реалистичному) значению. Если значение нас не устраивает, нужно изменить функцию от признаков $X_{i}$, например, отняв (если доля слишком велика) или прибавив (если доля слишком мала) некоторую константу к $h(X_{i})$.

Техническое примечание ⚡

Функция `scipy.stats.t.cdf` считает значение функции распределения для распределения Стьюдента с `df` степенями свободы в точке `x`.

Функция `np.random.binomial` генерирует `size` реализаций Биномиальных случайных величин с параметрами `n` и `p`, где параметр `p` в нашем случае различается для всех наблюдений и задается через условные вероятности.

In [ ]:
# Условная вероятность наличия у родителей высшего образования
parents_prob = scipy.stats.t.cdf(x = np.log(health + experience) - 4.3, df = 5)

# Факт наличия у родителей высшего образования
parents = np.random.binomial(n = 1, p = parents_prob, size = n)

# Первые несколько значений условных вероятностей и переменной
print(pd.DataFrame({'P(parents = 1|X)': np.round(parents_prob[0:10], 2),
                    'parents': parents[0:10]}))

# Доля индивидов, у которых родители с высшим образованием
print(pd.DataFrame(data    =  np.mean(parents),
                   index   = ['P(parents = 1)'],
                   columns = ['Оценка']))

Применим фантазию 🦄 и симулируем переменную воздействия: уровень образования индивида $\text{Educ}_{i}$.

Для краткости обозначим $X_{i}=(\text{Experience}_{i}, \text{Health}_{i})$ и $\tilde{X}_{i} = (\text{Experience}_{i}, \text{Health}_{i},\text{Abilities}_{i}, \text{Parents}_{i})$.

**Проблема** - в данном случае нельзя действовать по аналогии с переменной $\text{Parents}_{i}$, поскольку из содержательных соображений нужно также гарантировать, что $\text{Educ}_{i}$ будет связано не только с $\text{Experience}_{i}$ и $\text{Health}_{i}$, но и с $\text{Parents}_{i}$ и $\text{Abilities}_{i}$.

**Решение** - как бы мы не симулировали связь между $\text{Educ}_{i}$ и другими переменными, мы всегда (за исключением некоторых вырожденных случаев) можем предположить $g_{\text{Educ}}(X_{i}) = \text{E}(\text{Educ}_{i}|X_{i})$ и представить полученную модель в виде:

$$\text{Educ}_{i} = \text{E}(\text{Educ}_{i}|X_{i}) + \varepsilon_{i}^{\text{Educ}} = g_{\text{Educ}}(X_{i}) + \varepsilon_{i}^{\text{Educ}}$$

При этом нам не обязательно знать и симулировать саму функцию $g_{\text{Educ}}(X_{i})$, достаточно лишь того, что она будет существовать, поскольку совпадает с условным математическим ожиданием $\text{E}(\text{Educ}_{i}|X_{i})$, что гарантирует $\text{E}(\varepsilon_{i}^{\text{Educ}}|X_{i})=0$.

Таким образом, в симуляциях мы можем отталкиваться от вероятностей, условных на $\tilde{X}_{i}$, а не только на $X_{i}$.

$$P(\text{Educ}_{i} = 1|\tilde{X}_{i}) = \underbrace{F_{\text{Logistic}}(2\times\sqrt{\text{Abilities}_{i}+\text{Experience}_{i}+\text{Health}_{i} + 20\times\text{Parents}_{i}} - 19)}_{\text{из воображения}}$$

Где $F_{\text{Logistic}}$ - функция распределения стандартного логистического распределения.

In [ ]:
# Условная вероятность наличия у индивида высшего образования
educ_prob = scipy.stats.logistic.cdf(
    2 * np.sqrt(abilities + experience + health + 20 * parents) - 19)

# Факт наличия у индивида высшего образования
educ = np.random.binomial(n = 1, p = educ_prob, size = n)

# Первые несколько значений условных вероятностей переменной
print({'edyc': educ[0:10]})
print(pd.DataFrame({'P(educ = 1|X)': np.round(educ_prob[0:10], 2),
                    'educ': educ[0:10]}))

# Доля индивидов с высшим образованием
print(pd.DataFrame(data    =  np.mean(educ),
                   index   = ['P(educ= 1)'],
                   columns = ['Оценка']))

In [ ]:
# Убедимся в наличии корреляции между уровнем образования,
# способностями и образованием родителей
print(np.round(np.corrcoef([educ, parents, abilities, experience, health]), 2))

Применим фантазию 🦄 и для простоты **сперва** рассмотрим случай **без** эндогенности, то есть когда переменная $\text{Abilities}_{i}$ имеется в данных, а значит уравнение заработной платы можно рассматривать в форме:

$$\text{Wage}_{i} = \alpha \text{Educ}_{i} + g(\text{Experience}_{i}, \text{Health}_{i}, \text{Abilities}_{i}) + \varepsilon_{i}^{\text{Wage}}$$

Предположим (из воображения), что случайная ошибка $\varepsilon_{i}^{\text{Wage}}$ была получена из экспоненциального распределения с параметром $\lambda = 0.01$ и стандартизирована к нулевому математическому ожиданию.

Допустим, что неизвестная исследователю функция имеет вид:

$$g(\text{Experience}_{i}, \text{Health}_{i}, \text{Abilities}_{i}) = \underbrace{\text{Abilities}_{i} + 100\times\frac{5\times\text{Experience}_{i} - 0.1\times\text{Experience}_{i}^2}{100-\text{Health}_{i}}}_{\text{из воображения}}$$

In [ ]:
# Случайная ошибка
error_wage = scipy.stats.expon.rvs(size = n, scale = 10, loc = 0) - 10

# Отдача от образования
alpha = 10

# Функция от контрольных переменных
g = abilities + 100 * (5 * experience - 0.1 * experience ** 2) / (100 - health)

# Зарплата
wage = alpha * educ + g + error_wage

In [ ]:
# Аггрегируем данные в датафрейм
df = pd.DataFrame({'educ': educ, 'experience': experience,
                   'health': health, 'abilities': abilities,
                   'parents': parents, 'wage': wage})

# Посмотрим на симулированные данные
df.head(10).style.format(precision = 2)

Таким образом, мы самостоятельно (искусственно) создали (симулировали) данные. На практике у нас нет такой возможности и мы не знаем, как именно были получены данные, поэтому можем лишь предполагать выполнение предпосылок применяемых методов. Однако, в данном случае мы знаем, поскольку сами создали данные, что предпосылки выполняются, а именно:

$$E(\varepsilon^{\text{Wage}}_{i}|\text{Experience}_{i}, \text{Health}_{i},\text{Parents}_{i})=0$$

$$E(\varepsilon^{\text{Educ}}_{i}|\text{Experience}_{i}, \text{Health}_{i})=0$$

$$E(\varepsilon^{\text{Parents}}_{i}|\text{Experience}_{i}, \text{Health}_{i})=0$$

**Оценивание отдачи от образования с помощью МНК** 🐱

Цели ⭐

*   Оценить отдачу от образования, то есть параметр $\alpha$, с пощью метода наименьших квадратов (МНК).

Обычно, при оценивании связи между зависимой переменной и признаками исследователь предполагает линейное уравнение и оценивают его методом наименьших квадратов (МНК):

$$\text{Wage}_{i} = \beta_{0} + \alpha \text{Educ}_{i} + \beta_{1}\text{Experience}_{i} + \beta_{2}\text{Health}_{i} + \beta_{3}\text{Abilities}_{i} + \varepsilon_{i}$$

**Проблема** - если связи на самом деле нелинейные и плохо аппроксимируются линейной регрессией (как в нашей симуляции), то МНК оценка $\hat{\alpha}$ окажется несостоятельной и может обладать очень серьезным смещением.


Техническое примечание ⚡

Функция `sm.add_constant` добавляет константу в число признаков.

In [ ]:
# Разделим выборку на целевую и объясняющие переменные
y = df.loc[:, ['wage']]
x = df.loc[:, df.columns.drop(['wage', 'parents'])]
x = sm.add_constant(x)

Техническое примечание ⚡

Функция `sm.OLS(y, x).fit()` позволяет использовать линейный МНК. Метод `summary()` отвечает за красивую выдачу результатов.

Подробная информация может быть найдена в [документации](https://www.statsmodels.org/stable/gettingstarted.html).

In [ ]:
# Оценим отдачу от образования с помощью метода наименьших квадратов
ls = sm.OLS(y, x).fit()
print(ls.summary())

In [ ]:
# Сравним МНК оценку и истинное значение
# Сравним МНК и ДМО оценки
print(pd.DataFrame(data    = [alpha, ls.params["educ"]],
                   index   = ['Истина', 'МНК'],
                   columns = ['Альфа']))

**Решение** - воспользоваться двойным машинным обучением (DML), ослбив предпосылку о линейной связи:

$$\text{Wage}_{i} = \alpha \text{Educ}_{i} + g(\text{Experience}_{i}, \text{Health}_{i}, \text{Abilities}_{i}) + \varepsilon_{i}^{\text{Wage}}$$

**Оценивание отдачи от образования с помощью ДМО** 🐱

Цели ⭐

*   Оценить отдачу от образования, то есть параметр $\alpha$, с пощью двойного машинного обучения (ДМО).
*   Сравнить МНК и ДМО оценки.

Техническое примечание ⚡

Функция `dml.DoubleMLData()` позволяет подготовить данные к применению ДМО.

Основные аргументы:

*   `data` - исходные данные
*   `y_col` - название зависимой переменной $Y_{i}$
*   `d_col` - назвнаие переменной воздействия $T_{i}$
*   `x_cols`- названия контрольных переменных $X_{i}$

Дополнительная информация может быть найдена в [документации](https://docs.doubleml.org/stable/api/generated/doubleml.DoubleMLData.html).


In [ ]:
# Данные в формате, необходимом для применения DML
dml_standard_data = dml.DoubleMLData(
                            data = df,
                            y_col = 'wage',
                            d_cols = 'educ',
                            x_cols = ['experience', 'health', 'abilities'])

Напомним, что на первом шаге ДМО необходимо оценить условные математические ожидания $g_{Y}(X_{i}) = \text{E}(Y_{i}|X_{i})$ и $g_{T}(X_{i}) = \text{E}(T_{i}|X_{i})$ с помощью машинного обучения. Выберем два метода машинного обучения с заданными значениясми гиперпараметров.

**Примечание** - на практике гиперпараметры этих методов следует подобрать, например, с помощью кросс-валидации. Однако, для краткости опустим этот момент и выберем конкретные значения параметров, что, однако, может негативно сказаться на итоговой точности метода.

In [ ]:
# Метод оценивания E(Y | X)
g_Y = RandomForestRegressor(n_estimators = 100,
                            max_depth    = 20,
                            max_features = 3)

# Метод оценивания E(T | X)
g_T = RandomForestClassifier(n_estimators = 100,
                             max_depth    = 20,
                             max_features = 3)

Технический комментарий ⚡

Функция `DoubleMLPLR()` позволяет подготовиться к оцениваю ДМО.

Основные аргументы:


*   `obj_dml_data` - данные, предварительно созданные с помощью функции `DoubleMLData()`.
*  ` ml_l` - метод машинного обучения, используемый для оценивания $\text{E}(Y_{i}|X_{i})$.
*  ` ml_m` - метод машинного обучения, используемый для оценивания $\text{E}(T_{i}|X_{i})$.
*  `n_folds` - на сколько частей разбивается выборка при кросс-фиттинге. Чем больше, тем, обычно, точнее, но дольше оценивание.
*  `rep` - сколько раз повторить оценивание.

Дополнительная информация может быть найдена в [документации](https://docs.doubleml.org/stable/api/generated/doubleml.DoubleMLPLR.html).

In [ ]:
# Подготовка объекта
dml_standard = dml.DoubleMLPLR(obj_dml_data = dml_standard_data, # данные
                               ml_l = g_Y, ml_m = g_T,           # методы оценивания
                               n_rep = 1,                        # число повторений
                               n_folds = 5)                      # разибения выборки

In [ ]:
# Оценим параметры
dml_standard.fit()

In [ ]:
# Посмотрим на результат
print(dml_standard)

In [ ]:
# Сравним МНК и ДМО оценки
print(pd.DataFrame(data    = [alpha, ls.params["educ"], dml_standard.coef[0]],
                   index   = ['Истина', 'МНК', 'ДМО'],
                   columns = ['Альфа']))

**Оценивание отдачи от образования с помощью ДМО в условиях эндогенности** 🐱

Цели ⭐

*   Оценить отдачу от образования, то есть параметр $\alpha$, с пощью двойного машинного обучения (ДМО) без инструментальной переменной.
*   Оценить отдачу от образования, то есть параметр $\alpha$, с пощью двойного машинного обучения (ДМО) с инструментальной переменной.
*   Сравнить результаты с инструментальной переменной и без нее.

Допустим, что $\text{Abilities}_{i}$ не наблюдается в данных. Тогда уравнение зарплаты принимает вид:

$$\text{Wage}_{i} = \alpha \times \text{Educ}_{i} + g(\text{Experience}_{i}, \text{Health}_{i}) + v_{i}$$

Где новая случая ошибка $v_{i}$ "поглотила" в себя $\text{Abilities}_{i}$, поскольку способности теперь не наблюдаются в данных:

$$v_{i} = \varepsilon_{i}^{\text{Wage}} + \text{Abilities}_{i} - E(\text{Abilities}_{i}|\text{Experience}_{i},\text{Health}_{i},\text{Parents}_{i})$$

$$g(\text{Experience}_{i}, \text{Health}_{i}) = g(\text{Experience}_{i}, \text{Health}_{i},\text{Abilities}_{i}) - \text{Abilities}_{i} + \text{E}(\text{Abilities}_{i}|\text{Experience}_{i},\text{Health}_{i},\text{Parents}_{i})$$

Где $\text{E}(\text{Abilities}_{i}|\text{Experience}_{i},\text{Health}_{i},\text{Parents}_{i}) = \text{E}(\text{Abilities}_{i})$ в силу того, что в нашем примере $\text{Abilities}_{i}$ не зависит от $\text{Experience}_{i}$, $\text{Health}_{i}$ и $\text{Parents}_{i}$.

**Примечание** - убедитесь самостоятельно, что:

$$\text{E}(v_{i}|\text{Experience}_{i},\text{Health}_{i},\text{Parents}_{i}) = 0$$

Поскольку $\text{Abilities}_{i}$ коррелирует с $\text{Educ}_{i}$, то и $v_{i}$ коррелирует с $\text{Educ}_{i}$. Следовательно возникает проблема эндогенности.

In [ ]:
# Корреляция способностей с образованием и
# случайной ошибки с образованием
print(pd.DataFrame(data    = [np.corrcoef(abilities, educ)[0, 1],
                              np.corrcoef(error_wage + abilities, educ)[0, 1]],
                   index   = ['cor(abilities, educ)', 'cor(v, educ)'],
                   columns = ['Оценка']))

Попробуем сперва воспользоваться обычным ДМО, то есть без инструментальных переменных. Для этого просто исключим `abilities` из аргумента `x_cols` функции `dml.DoubleMLData()`.

In [ ]:
# Воспользуемся DML методом на данных, в
# которых нет информации о способностях

# Подготовим данные
dml_standard2_data = dml.DoubleMLData(
                             data = df,
                             y_col = 'wage',
                             d_cols = 'educ',
                             x_cols = ['experience', 'health'])

# Подготовка объекта
dml_standard2 = dml.DoubleMLPLR(obj_dml_data = dml_standard2_data,
                                ml_l = g_Y, ml_m = g_T,
                                n_rep = 1,
                                n_folds = 5)

# Оценим параметры
dml_standard2.fit()

# Посмотрим на результат
print(dml_standard2)

In [ ]:
# Посмотрим на результат оценивания
print(pd.DataFrame(data    = [alpha, dml_standard2.coef[0]],
                   index   = ['Истина', 'ДМО без инструмента'],
                   columns = ['Альфа']))

Учтем эндогенность, воспользовавшись ДМО методом с инструментальной переменной $\text{Parents}_{i}$. При этом мы знаем, что инструмент $\text{Parents}_{i}$ является экзогенным. Однако, на практике, используя реальные данные, мы не можем никак этого проверить, даже тестами, поэтому вынуждены опираться исключительно на содержательные (словестные) экономические соображения.

Отметим, что с содержательной точки зрения можно оспорить экзогенность $\text{Parents}_{i}$, поскольку, например, родители, обладающие более высоким уровнем образования, могли больше вкладываться в развитие своего ребенка, что повлияло на его способности $\text{Abilities}_{i}$.

In [ ]:
# Убедимся в экзогенности инструмента (на реальных данных недоступно)
print(pd.DataFrame(data    = np.corrcoef(parents, abilities)[0, 1],
                   index   = ['cor(parents, abilities)'],
                   columns = ['Оценка']))

Техническое примечание ⚡

Чтобы добавить инструментальную переменную, необходимо указать ее название в качестве значения параметра `z_cols` функции `DoubleMLData()`.

Для оценивания ДМО с инструментальными переменными используется функция `DoubleMLPLIV()`, в которой метод оценивания $\text{E}(Z_{i} | X_{i})$ подается через аргумент `ml_r`. Подробная информация об этой функции может быть найдена в [документации](https://docs.doubleml.org/stable/api/generated/doubleml.DoubleMLPLIV.html).

In [ ]:
# Представим, что у исследователя нет
# информации о способностях индивида и
# применим метод инструментальных переменных

# Подготовим данные
dml_iv_data = dml.DoubleMLData(data = df,
                               y_col = 'wage',
                               d_cols = 'educ',
                               z_cols = 'parents',
                               x_cols = ['experience', 'health'])

# Метод оценивания E(Z | X)
g_Z = GradientBoostingClassifier(loss = 'log_loss',
                                 n_estimators = 100,
                                 learning_rate = 0.1)

# Подготовка объекта
dml_iv = dml.DoubleMLPLIV(obj_dml_data = dml_iv_data,
                          ml_l = g_Y, ml_m = g_Z, ml_r = g_T,
                          n_rep = 1,
                          n_folds = 5)

# Оценим параметры
dml_iv.fit()

# Посмотрим на результат
print(dml_iv)

In [ ]:
# Сравним результаты до и после учета эндогенности
# Посмотрим на результат после учета эндогенности
print(pd.DataFrame(data    = [alpha, dml_standard2.coef[0], dml_iv.coef[0]],
                   index   = ['Истина', 'ДМО без инструмента',
                              'ДМО с инструментом'],
                   columns = ['Альфа']))

**Важно** - без учета эндогенности мы переоцениваем отдачу от образования $\alpha$: оценка слишком отклоняется от истинного значения в большую сторону. Интуитивно это обусловлено тем, что не учитывая эндогенность мы смешиваем два положительных эффекта: от образования $\text{Educ}_{i}$ и от способностей $\text{Abilities}_{i}$. Если бы образование отрицательно коррелировало со способностями или способности отрицательно влияли на заработную плату, то без использования инструментальных переменных мы бы серьезно недооценили отдачу от высшего образования.

**Вывод** - разница в оценках $\alpha$ без учета и с учетом эндогенности, при допущении об экзогенности инструмента, позволяет выдвинуть предположения о том, как ненаблюдаемые характеристики, порождающие эндогенность (способности), связаны с целевой переменной (зарплата) и переменной воздействия (образование).

**Тюнинг гиперпараметров** 🐱

Точность оценки $\alpha$ с помощью ДМО зависит от точностей оценок условных математических ожиданий $\hat{\text{E}}(Y_{i}|X_{i})$, $\hat{\text{E}}(T_{i}|X_{i})$ и $\hat{\text{E}}(Z_{i}|X_{i})$. Поэтому, тюнинг (подбор оптимальных значений) гиперпараметров методов машинного обучения, используемых для получения этих оценок, может существенно повысить точности оценки $\alpha$.

С технической точки зрения Тюнинг можно осуществить двумя способами:

*   Самостоятельно осуществить тюнинг гиперпараметров, а затем подставить в качестве аргументов `ml_l`, `ml_m` и `ml_r` модели с оптимальными значеинями гиперпараметров.
*   Воспользоваться методом `tune()`, который позволяет подобрать гиперпараметры после оценивания модели.

Рассмотрим второй способ.



In [ ]:
# Создадим новую модель, скопировав старую
dml_iv_tune = deepcopy(dml_iv)

Техническое примечание ⚡

Метод `tune()` позволяет подобрать оптимальные гиперпараметры. Если `search_mode = 'grid_search'`, то подбор гиперпараметров осуществляется с помощью ранее рассматривавшеся нами функции `sklearn.model_selection.GridSearchCV()`. Поэтому, достаточно указать перебираемые комбинации гиперпараметров для каждого метода в форме листа.

Аргумент `n_folds_tune` аналогичен аргументу `cv` функции `GridSearchCV()`, то есть определяет количество фолдов, используемых при кросс-валидации.

Подробная информация может быть найдена в [документации](https://docs.doubleml.org/stable/api/generated/doubleml.DoubleMLPLIV.html#doubleml.DoubleMLPLIV.tune) (нужно пролистать достаточно далеко вниз), а также в [примерах](https://docs.doubleml.org/stable/guide/learners.html#).

In [ ]:
# Перебираемые значения гиперпараметров
tune_grid = {'ml_l': {'max_depth': [5, 20, 100],    # параметры метода ml_l
                      'max_features': [2, 3]},      # E(Y|X)
             'ml_r': {'max_depth': [5, 20, 100],    # параметры метода ml_r
                      'max_features': [2, 3]},      # E(T|X)
             'ml_m': {'n_estimators': [10, 100],    # параметры метода ml_m
                      'learning_rate': [0.1, 1]}}   # E(Z|X)

In [ ]:
# Тюнинг (может осуществляться очень долго)
dml_iv_tune.tune(tune_grid, search_mode = 'grid_search', n_folds_tune = 2)

In [ ]:
# Посмотрим на оптимальные параметры, которые после тюнинга
# автоматически были установлены в качестве используемых при оценивании
dml_iv_tune.params
# Можно сделать так, что оптимальные параметры будут различаться между фолдами,
# поэтому каждая строка повторяется n_folds раз

In [ ]:
# Обучим заново модель
dml_iv_tune.fit()

# Посмотрим на результат
print(dml_iv_tune)

In [ ]:
# Сравним результаты до и после Тюнинга
print(pd.DataFrame(data    = [alpha, dml_standard2.coef[0],
                              dml_iv.coef[0], dml_iv_tune.coef[0]],
                   index   = ['Истина', 'ДМО без инструмента',
                              'ДМО с инструментом',
                              'Тюнингованный ДМО с инструментом'],
                   columns = ['Альфа']))

**Использование параметрического подхода** 🐱

Подключим пакеты

In [ ]:
# !pip install rpy2
# ! add-apt-repository -y ppa:cran/imagemagick
# ! apt-get update
# ! apt-get install -y libmagick++-dev
# ! R -e "install.packages('switchSelection')"

In [ ]:
import rpy2                                               # R в python
import rpy2.robjects as ro
import rpy2.robjects.packages as rpackages
from rpy2.robjects.vectors import StrVector
from rpy2.robjects.packages import importr, data
from rpy2.robjects import pandas2ri
from rpy2.robjects import IntVector, Formula

Установим необходимые библиотеки из `R`

In [ ]:
# Пакет для установки пакетов из R
utils = rpackages.importr('utils')
utils.chooseCRANmirror(ind = 1)

# Пакеты с базовым функционалом R
base  = rpackages.importr('base')
stats = rpackages.importr('stats')

# Подключение пакета
switchSelection = importr('switchSelection')

# Активация конвертации
pandas2ri.activate()

# Во избежание проблем с версиями pandas
pd.DataFrame.iteritems = pd.DataFrame.items

Часто эндогенность бывает удобно учесть в явном виде, предположив параметрическую модель. В рамках данных моделей вводятся достаточно сильные предпосылки о совместном нормальном распределении случайных ошибок и о линейной по параметрам форме, в которой регрессоры входят в уравнение. Однако, часто эти модели оказываются устойчивы к нарушению подобных предпосылок. На практике не редко бывает достаточно лишь включить квадратные и перекрестные члены между переменными, для того чтобы точность аппроксимации оказалась достаточно высокой.

$$\text{Wage}_{i} = \alpha \text{Educ}_{i} + \beta_{0} + \beta_{1}\text{Experience}_{i} + ... + \beta_{m} \text{Experience}_{i} * \text{Health}_{i} + \varepsilon_{i}$$

$$\text{Educ}_{i}^{*} = \gamma_{0} + \gamma_{1}\text{Experience}_{i} + ... + \gamma_{t} \text{Experience}_{i} * \text{Health}_{i} + \gamma_{t+1} \text{Parents}_{i} + u_{i}$$

$$\text{Educ}_{i} = \begin{cases}1\text{, если }\text{Educ}_{i}^{*}\geq0\\0\text{, в противном случае}\end{cases}$$

$$(\varepsilon_{i}, u_{i})\sim \text{N}\left(\begin{bmatrix}0\\0\end{bmatrix}, \begin{bmatrix}\sigma^2 & \rho\sigma\\ \rho\sigma & 1\end{bmatrix}\right)$$

Параметры данной системы оцениваются методом максимального правдоподобия (функция правдоподобия опускается для краткости).

In [ ]:
# Оценивание параметрической модели
formula_educ = Formula('educ ~ experience + health + parents + '
                       'I(experience ^ 2) + I(health ^ 2) + I(experience * health)')
formula_wage = Formula('wage ~ educ + experience + health + '
                       'I(experience ^ 2) + I(health ^ 2) + I(experience * health)')
model = switchSelection.msel(formula  = formula_educ,
                             formula2 = formula_wage,
                             data     = df)
print(base.summary(model))

Также, можно воспользоваться двухшаговой полупараметрической процедурой, аппроксимировав условное математическое ожидание $\text{E}\left(\varepsilon_{i}^{\text{Wage}}|\text{Educ}_{i}, \text{Experience}_{i}, \text{Health}\right)$ с помощью полинома степени `degrees`.

In [ ]:
# Оценивание полупараметрической модели
model2 = switchSelection.msel(formula   = formula_educ,
                              formula2  = formula_wage,
                              data      = df,
                              degrees   = 3,
                              estimator = '2step')
print(base.summary(model2))

In [ ]:
# Сравним точность оценок
print(pd.DataFrame(data    = [alpha, dml_standard2.coef[0],
                              dml_iv.coef[0], dml_iv_tune.coef[0],
                              stats.coef(model, type = "coef2", regime = 0)[1],
                              stats.coef(model2, type = "coef2", regime = 0)[1]],
                   index   = ['Истина', 'ДМО без инструмента',
                              'ДМО с инструментом',
                              'Тюнингованный ДМО с инструментом',
                              'Параметрическая модель',
                              'Полупараметрическая модель'],
                   columns = ['Альфа']))